In [0]:
# NOTEBOOK   : silver_order_items
# PURPOSE    : Clean and transform bronze.order_items table
# SOURCE     : fincompliance_catalog.bronze.order_items
# DESTINATION: fincompliance_catalog.silver.order_items
# TRANSFORMATIONS:
#   - Deduplicate on order_id and order_item_id
#   - Cast shipping_limit_date to TimestampType
#   - Round price to 2 decimal places
#   - Round freight_value to 2 decimal places
#   - Calculate total_item_value = price + freight_value
#   - Calculate freight_percentage of total
#   - Classify items by price range
#   - Drop bronze audit columns
#   - Add silver_updated_at audit column

In [0]:
%run ../configs/config

In [0]:
## IMPORT PYSPARK FUNCTIONS
from pyspark.sql.functions import (
    col,               # refers to a column by name
    to_timestamp,      # converts string to timestamp
    round as spark_round, # rounds decimal numbers
    when,              # if/else conditions
    lit,               # converts python value to spark column
    current_timestamp, # gets current date and time
    sum as spark_sum,  # adds up values
    avg,               # calculates average
    count              # counts rows
)

In [0]:
# READ FROM BRONZE
# Read the raw order_items table from bronze layer
# Show data BEFORE transformation


df_order_items = spark.table(f"{BRONZE_DB}.order_items")

print("=" * 60)
print("BRONZE.ORDER_ITEMS — Before Transformation")
print("=" * 60)
print(f"Total rows    : {df_order_items.count():,}")
print(f"Total columns : {len(df_order_items.columns)}")

# Show column names and data types
print("\nColumn names and data types:")
for field in df_order_items.schema.fields:
    print(f"  → {field.name:<40} : {field.dataType}")

# Show sample data
print("\nSample data BEFORE transformation (3 rows):")
df_order_items.show(3, truncate=True)

# Show price statistics
print("\nPrice statistics:")
df_order_items \
    .select("price", "freight_value") \
    .summary("min", "max", "mean", "count") \
    .show()

In [0]:
# DEDUPLICATION
# order_items has a COMPOSITE PRIMARY KEY
# order_id + order_item_id TOGETHER make a unique row
# NOT just order_id alone

# Count rows BEFORE deduplication
rows_before = df_order_items.count()
print(f"Rows BEFORE deduplication : {rows_before:,}")

# Apply deduplication on COMPOSITE KEY
# Both columns together identify a unique item
df_order_items_silver = df_order_items \
    .dropDuplicates(["order_id", "order_item_id"])

# Count rows AFTER deduplication
rows_after = df_order_items_silver.count()
print(f"Rows AFTER deduplication  : {rows_after:,}")

duplicates_removed = rows_before - rows_after
print(f"Duplicates removed : {duplicates_removed:,}")

if duplicates_removed == 0:
    print("No duplicates found")
else:
    print(f"{duplicates_removed:,} duplicates removed")

# Show order_item_id distribution
# This tells us how many items per order
print("\nItems per order distribution:")
df_order_items_silver \
    .groupBy("order_item_id") \
    .count() \
    .orderBy("order_item_id") \
    .show(10, truncate=False)



In [0]:
# ROUND PRICE AND FREIGHT VALUE
# Round to 2 decimal places
# Monetary values must always be precise

df_order_items_silver = df_order_items_silver \
    .withColumn("price",
                spark_round(col("price"), 2)) \
    .withColumn("freight_value",
                spark_round(col("freight_value"), 2))

print("Price and freight rounded to 2 decimal places")
df_order_items_silver \
    .select("price", "freight_value") \
    .show(5, truncate=False)
print("Rounding complete")

In [0]:
# CALCULATE TOTAL ITEM VALUE
# total_item_value = price + freight_value
# This column DID NOT EXIST in bronze
# Total cost customer pays per item

df_order_items_silver = df_order_items_silver \
    .withColumn(
        "total_item_value",
        spark_round(
            col("price") + col("freight_value"), 2
        )
    )

print("Total item value calculated:")
df_order_items_silver \
    .select(
        "order_id",
        "price",
        "freight_value",
        "total_item_value"
    ) \
    .show(5, truncate=True)
print("Total item value calculated")

In [0]:
# CALCULATE FREIGHT PERCENTAGE
# What percentage of total cost is shipping?
# freight_value / total_item_value * 100
# This column DID NOT EXIST in bronze

df_order_items_silver = df_order_items_silver \
    .withColumn(
        "freight_percentage",
        spark_round(
            (col("freight_value") /
             col("total_item_value")) * 100, 2
        )
    )

print("Freight percentage calculated:")
df_order_items_silver \
    .select(
        "price",
        "freight_value",
        "total_item_value",
        "freight_percentage"
    ) \
    .show(5, truncate=True)

# Show average freight percentage
avg_freight_pct = df_order_items_silver \
    .select(avg("freight_percentage")) \
    .collect()[0][0]
print(f"\nAverage freight percentage : {avg_freight_pct:.1f}%")
print("Freight percentage calculated")

In [0]:
# CLASSIFY PRICE RANGE
# Classify items into price categories
# Budget    : price < 50
# Mid Range : price 50 to 200
# Premium   : price > 200
# This column DID NOT EXIST in bronze

df_order_items_silver = df_order_items_silver \
    .withColumn(
        "price_range",
        when(col("price") < 50, "Budget")
        .when(
            (col("price") >= 50) &
            (col("price") <= 200), "Mid Range")
        .when(col("price") > 200, "Premium")
        .otherwise("Unknown")
    )

print("Price range classification:")
df_order_items_silver \
    .groupBy("price_range") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)
print("Price range classification complete")

In [0]:
# FREE SHIPPING FLAG
# Identify items with free shipping
# freight_value = 0 → is_free_shipping = 1
# This column DID NOT EXIST in bronze

df_order_items_silver = df_order_items_silver \
    .withColumn(
        "is_free_shipping",
        when(col("freight_value") == 0, 1)
        .otherwise(0)
    )

free_shipping = df_order_items_silver \
    .filter(col("is_free_shipping") == 1) \
    .count()
total = df_order_items_silver.count()

print(f"Free shipping items : {free_shipping:,}")
print(f"Paid shipping items : {total - free_shipping:,}")
print(f"Free shipping rate  : {free_shipping/total*100:.1f}%")
print("Free shipping flag added")


In [0]:
# ============================================================
# DROP BRONZE AUDIT COLUMNS
# ============================================================

df_order_items_silver = df_order_items_silver \
    .drop("ingestion_timestamp", "source_file")

print("Bronze audit columns dropped")
print(f"Total columns now : {len(df_order_items_silver.columns)}")
print("Done")

In [0]:
# ============================================================
# ADD SILVER AUDIT COLUMN
# ============================================================

df_order_items_silver = df_order_items_silver \
    .withColumn("silver_updated_at", current_timestamp())

print("Final columns in silver.order_items:")
for col_name in df_order_items_silver.columns:
    print(f"  → {col_name}")
print(f"\nTotal rows    : {df_order_items_silver.count():,}")
print(f"Total columns : {len(df_order_items_silver.columns)}")
print("Silver audit column added")

In [0]:
# ============================================================
# CELL 14 - WRITE TO SILVER
# ============================================================

print("Writing to silver.order_items...")

(
    df_order_items_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.order_items")
)

print(f"Written to {SILVER_DB}.order_items")
print(f"Rows    : {df_order_items_silver.count():,}")
print(f"Columns : {len(df_order_items_silver.columns)}")

In [0]:
# ============================================================
# CELL 15 - VERIFICATION
# ============================================================

print("=" * 60)
print("SILVER.ORDER_ITEMS VERIFICATION")
print("=" * 60)

df_silver = spark.table(f"{SILVER_DB}.order_items")

# CHECK 1 — Row count
bronze_count = spark.table(f"{BRONZE_DB}.order_items").count()
silver_count = df_silver.count()
print(f"\nCHECK 1 — Row counts:")
print(f"  Bronze : {bronze_count:,}")
print(f"  Silver : {silver_count:,}")
if bronze_count == silver_count:
    print(f"Row counts match")
else:
    print(f"Difference : {bronze_count - silver_count:,}")

# CHECK 2 — New columns
print(f"\nCHECK 2 — New columns:")
new_cols = [
    "total_item_value",
    "freight_percentage",
    "price_range",
    "is_free_shipping",
    "silver_updated_at"
]
for c in new_cols:
    if c in df_silver.columns:
        print(f"{c:<30} exists")
    else:
        print(f"{c:<30} MISSING")

# CHECK 3 — Price range breakdown
print(f"\nCHECK 3 — Price range breakdown:")
df_silver \
    .groupBy("price_range") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# CHECK 4 — Free shipping
print(f"CHECK 4 — Free shipping:")
df_silver \
    .groupBy("is_free_shipping") \
    .count() \
    .show(truncate=False)

# CHECK 5 — Value statistics
print(f"CHECK 5 — Value statistics:")
df_silver \
    .select(
        "price",
        "freight_value",
        "total_item_value",
        "freight_percentage"
    ) \
    .summary("min", "max", "mean") \
    .show()

print("=" * 60)
print("silver.order_items verification complete!")
print(f"Rows  : {silver_count:,}")
print(f"Cols  : {len(df_silver.columns)}")
print("=" * 60)